<a href="https://colab.research.google.com/github/syedabusafwan/ML-practice/blob/main/Safwan_WEBCam_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics opencv-python -q

import cv2
import time
from ultralytics import YOLO

In [ ]:
model = YOLO("yolov8n.pt")

In [ ]:
video_path = "/content/drive/MyDrive/Colab Notebooks/WhatsApp Video 2026-03-20 at 2.31.44 AM.mp4"
cap = cv2.VideoCapture(video_path)

In [ ]:
# 4. Output Video Setup (LOW RES)
frame_width = 640
frame_height = 480
fps = 30

out = cv2.VideoWriter(
    "output.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (frame_width, frame_height)
)


In [ ]:
# 5. Time Variables (AuraGuard Logic)
person_last_seen = time.time()
phone_start_time = None
last_drink_time = time.time()

PERSON_TIMEOUT = 5
PHONE_THRESHOLD = 2
HYDRATION_THRESHOLD = 30

# 6. Frame Control (MEMORY FIX)
frame_count = 0

while True:

    ret, frame = cap.read()
    if not ret:
        break


    # 🔥 Resize frame (VERY IMPORTANT)
    frame = cv2.resize(frame, (frame_width, frame_height))

    # YOLO detection
    results = model(frame)[0]

    detected_classes = []

    # ===============================
    # Draw detections
    # ===============================
    for box in results.boxes:
        cls = int(box.cls[0])
        detected_classes.append(cls)

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        label = model.names[cls]

        cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
        cv2.putText(frame,label,(x1,y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX,0.4,(0,255,0),1)

    # ===============================
    # EMPTY DESK
    # ===============================
    if 0 not in detected_classes:
        if time.time() - person_last_seen > PERSON_TIMEOUT:
            cv2.putText(frame,
            "System Paused: User Away",
            (10,20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,(0,255,255),2)

            out.write(frame)
            continue
    else:
        person_last_seen = time.time()

    # ===============================
    # PHONE DISTRACTION
    # ===============================
    if 67 in detected_classes:

        if phone_start_time is None:
            phone_start_time = time.time()

        if time.time() - phone_start_time > PHONE_THRESHOLD:

            cv2.putText(frame,
            "WARNING: PUT PHONE AWAY",
            (10,40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,(0,0,255),2)

    else:
        phone_start_time = None

    # ===============================
    # HYDRATION
    # ===============================
    if 39 in detected_classes or 41 in detected_classes:
        last_drink_time = time.time()

    if time.time() - last_drink_time > HYDRATION_THRESHOLD:

        cv2.putText(frame,
        "Drink Water!",
        (10,60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,(255,0,0),2)

    # ===============================
    # FOCUS MODE
    # ===============================
    if 0 in detected_classes and 67 not in detected_classes:
        if time.time() - last_drink_time < HYDRATION_THRESHOLD:

            cv2.putText(frame,
            "Status: Focusing",
            (10,80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,(0,255,0),2)

    # ===============================
    # Save frame
    # ===============================
    out.write(frame)

# 8. Release Resources
cap.release()
out.release()

print("✅ Output video saved as output.mp4")

Streaming output truncated to the last 5000 lines.
Speed: 1.7ms preprocess, 7.5ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 cup, 1 dining table, 2 laptops, 1 keyboard, 1 book, 6.8ms
Speed: 1.7ms preprocess, 6.8ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 cup, 1 dining table, 2 laptops, 1 keyboard, 1 book, 7.0ms
Speed: 1.7ms preprocess, 7.0ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 cup, 1 dining table, 2 laptops, 1 keyboard, 1 book, 7.2ms
Speed: 1.6ms preprocess, 7.2ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 cup, 1 dining table, 2 laptops, 1 keyboard, 1 book, 7.1ms
Speed: 2.3ms preprocess, 7.1ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 cup, 1 dining table, 2 laptops, 1 keyboard, 1 book, 7.1ms
Speed: 1.5ms preprocess, 7.1ms inference, 1.1ms postprocess per image at shape 

In [ ]:
from IPython.display import Video
Video("output.mp4")